# Synthetic Tabular Data Generation - Staged Optimization Driver

**Enhanced notebook for synthetic tabular data generation with staged hyperparameter optimization.**

This notebook provides a staged optimization workflow with:
- Section 1: Setup and imports
- Section 2: Data loading, preprocessing, and EDA
- Section 3: Demo models with default parameters
- Section 4: **Staged Hyperparameter Optimization** (NEW)
  - 4.1: Configuration
  - 4.2: Pilot phase (15 trials) with time estimates
  - 4.3: Continuation options
  - 4.4: Diminishing returns analysis
  - 4.5: Additional batches (optional)
- Section 5: Final models with best parameters

Based on STG-Driver-breast-cancer.ipynb with staged optimization enhancements.

## 1 Setup and Configuration

In [1]:
# Code Chunk ID: CHUNK_1_0_0_A - Import Setup Module
# Import all functionality from setup.py
import sys
sys.path.insert(0, "/home/ec2-user/SageMaker/tableGenCompare")

from setup import *

# Refresh session timestamp to current date
refresh_session_timestamp()

# ============================================================================
# FRESH START CONTROL - Set to True to wipe all checkpoints and start over
# ============================================================================
FRESH_START = True   # <-- Change to True to clear ALL saved progress
# ============================================================================

print("SETUP MODULE IMPORTED SUCCESSFULLY!")
print(f"FRESH_START = {FRESH_START}")
print("=" * 60)

[OK] Essential data science libraries imported successfully!
[OK] Model registry loaded from src/models/registry.py
[OK] Batch training module loaded from src/models/batch_training.py
[OK] Search spaces module loaded from src/models/search_spaces.py
[OK] Batch optimization module loaded from src/models/batch_optimization.py
[OK] Staged optimization module loaded from src/models/staged_optimization.py
[CONFIG] Session timestamp: 2026-03-16
[OK] Parameter management functions loaded from src/utils/parameters.py
[OK] Hyperparameter optimization evaluation functions loaded from src/evaluation/hyperparameters.py
[OK] Optuna objective functions for PATE-GAN and MEDGAN loaded (Phase 5)
Detected sklearn 1.7.2 - applying compatibility patch...
INFO: Applying sklearn compatibility patches for version 1.7.2
Global sklearn compatibility patch applied successfully
CTAB-GAN imported successfully
[OK] CTAB-GAN+ detected and available
[OK] GANerAidModel imported successfully from src.models.implementa

## 2 Data Loading and Pre-processing

### 2.1 Configuration

**USER ATTENTION NEEDED**: Edit the `NOTEBOOK_CONFIG` dictionary below to match your dataset.

In [2]:
# Code Chunk ID: CHUNK_2_1_1_A - NOTEBOOK CONFIGURATION
# ============================================================================
# USER CONFIGURATION - Edit ONLY this block for your dataset
# ============================================================================

NOTEBOOK_CONFIG = {
    # ========== REQUIRED: Dataset Settings ==========
    "data_file": "data/Pakistani_Diabetes_Dataset.csv",  # Path to your CSV file
    "target_column": "Outcome",                 # Target/outcome column name

    # ========== OPTIONAL: Dataset Metadata ==========
    "dataset_name": "Pakistani Diabetes Dataset",      # Display name
    "dataset_identifier_override": None,          # Override auto-detected ID (or None)

    # ========== OPTIONAL: Column Configuration ==========
    # Binary 0/1 flags in this dataset (7 features):
    "categorical_columns": ["Gender", "Rgn ", "his", "vision", "dipsia", "uria", "neph"],
    "task_type": "classification",

    # ========== OPTIONAL: Fairness Evaluation ==========
    # Protected attribute for fairness metrics (use cleaned column name after preprocessing).
    # Gender is binary (0/1) and survives preprocessing as-is.
    "protected_col": "gender",

    # ========== OPTIONAL: Data Subsetting ==========
    "use_row_subset": False,                      # 912 rows - small enough to use all
    "sample_n": 500,                              # Number of rows to sample
    "sample_random_state": 42,                    # Random seed for reproducibility

    # ========== OPTIONAL: Missing Data Handling ==========
    "missing_strategy": "none",                   # No missing values in this dataset
    "mice_max_iter": 10,                          # Max iterations for MICE imputation

    # ========== OPTIONAL: Model Selection ==========
    "models_to_run": ["ctgan", "tvae", "copulagan", "ctabgan", "ctabganplus", "pategan", "medgan"],     # "all" or list like ["ctgan", "tvae"]
    # "models_to_run": ["ctgan", "tvae", "copulagan", "ctabgan", "ctabganplus", "ganeraid", "pategan", "medgan"],  

    # ========== OPTIONAL: Tuning Configuration ==========
    "tuning_mode": "full",                        # "smoke" (quick) | "full" (thorough)
    "n_trials_smoke": 15,                          # Trials for smoke testing / pilot phase
}

print("NOTEBOOK_CONFIG set successfully!")
print(f"  Data file: {NOTEBOOK_CONFIG['data_file']}")
print(f"  Target column: {NOTEBOOK_CONFIG['target_column']}")
print(f"  Protected column: {NOTEBOOK_CONFIG.get('protected_col', None)}")
print(f"  Models to run: {NOTEBOOK_CONFIG['models_to_run']}")
print(f"  Tuning mode: {NOTEBOOK_CONFIG['tuning_mode']}")

NOTEBOOK_CONFIG set successfully!
  Data file: data/Pakistani_Diabetes_Dataset.csv
  Target column: Outcome
  Protected column: gender
  Models to run: ['ctgan', 'tvae', 'copulagan', 'ctabgan', 'ctabganplus', 'pategan', 'medgan']
  Tuning mode: full


### 2.2 Load and Preprocess Data

In [3]:
# Code Chunk ID: CHUNK_2_1_2_A - LOAD AND PREPROCESS DATA
# ============================================================================
# This uses the config-driven preprocessing pipeline
# ============================================================================

if not FRESH_START and 'checkpoint' in dir() and hasattr(checkpoint, 'exists') and checkpoint.exists('section_2'):
    saved = checkpoint.load('section_2')
    data = saved['data']
    original_data = saved['original_data']
    target_column = saved['target_column']
    DATASET_IDENTIFIER = saved['DATASET_IDENTIFIER']
    categorical_columns = saved['categorical_columns']
    metadata = saved['metadata']
    models_to_run = saved['models_to_run']
    RUN_MODE = saved['RUN_MODE']
    TARGET_COLUMN = target_column
    print("[RESUME] Loaded Section 2 from checkpoint")
else:
    # Load and preprocess data using the config
    (
        data,                  # Processed DataFrame
        original_data,         # Copy for reference
        target_column,         # Target column name (cleaned)
        DATASET_IDENTIFIER,    # Dataset identifier for results paths
        categorical_columns,   # List of categorical columns
        metadata               # Full preprocessing metadata
    ) = load_and_preprocess_from_config(NOTEBOOK_CONFIG)

    # Set aliases for backward compatibility with existing notebook cells
    TARGET_COLUMN = target_column

    # Get models to run based on config
    models_to_run = get_models_to_run(NOTEBOOK_CONFIG)

    # Set RUN_MODE for backward compatibility with Section 4 tuning cells
    RUN_MODE = "debug" if NOTEBOOK_CONFIG['tuning_mode'] == "smoke" else "full"

# Initialize checkpoint system (now that DATASET_IDENTIFIER is known)
checkpoint = SectionCheckpoint(DATASET_IDENTIFIER)

# If FRESH_START, wipe all checkpoints and optimization studies
if FRESH_START:
    n_removed = checkpoint.clear_all()
    print(f"[FRESH START] Cleared {n_removed} checkpoint(s) and optimization studies")

existing = checkpoint.list_checkpoints()
if existing:
    print(f"[CHECKPOINT] Found {len(existing)} existing checkpoint(s): {existing}")

# Save Section 2 checkpoint (idempotent - overwrites if exists)
if not checkpoint.exists('section_2'):
    checkpoint.save('section_2', {
        'data': data, 'original_data': original_data,
        'target_column': target_column, 'DATASET_IDENTIFIER': DATASET_IDENTIFIER,
        'categorical_columns': categorical_columns, 'metadata': metadata,
        'models_to_run': models_to_run, 'RUN_MODE': RUN_MODE,
    })

print()
print("=" * 60)
print("PREPROCESSING COMPLETE")
print("=" * 60)
print(f"  Dataset identifier: {DATASET_IDENTIFIER}")
print(f"  Processed shape: {data.shape}")
print(f"  Target column: {target_column}")
print(f"  Task type: {metadata['task_type']}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Models to run: {models_to_run}")
print(f"  RUN_MODE: {RUN_MODE}")
print(f"  Session: {SESSION_TIMESTAMP}")
print(f"  Results path: {get_results_path(DATASET_IDENTIFIER, 2)}")
print("=" * 60)

[LOAD] Loading data from: data/Pakistani_Diabetes_Dataset.csv
[LOAD] Loaded 912 rows, 19 columns
[PREPROCESS] Starting preprocessing pipeline
[PREPROCESS] Input shape: (912, 19)
[PREPROCESS] Categorical columns: ['gender', 'rgn', 'his', 'vision', 'dipsia', 'uria', 'neph']
[PREPROCESS] Task type: classification
[PREPROCESS] Final shape: (912, 19)
[PREPROCESS] Dataset identifier: pakistani-diabetes-dataset
[FLUSH] Removed 14 pickle file(s) from results/pakistani-diabetes-dataset/optimization_studies
[FRESH START] Cleared 17 checkpoint(s) and optimization studies

PREPROCESSING COMPLETE
  Dataset identifier: pakistani-diabetes-dataset
  Processed shape: (912, 19)
  Target column: outcome
  Task type: classification
  Categorical columns: ['gender', 'rgn', 'his', 'vision', 'dipsia', 'uria', 'neph']
  Models to run: ['ctgan', 'tvae', 'copulagan', 'ctabgan', 'ctabganplus', 'pategan', 'medgan']
  RUN_MODE: full
  Session: 2026-03-16
  Results path: results/pakistani-diabetes-dataset/2026-03-1

### 2.3 Exploratory Data Analysis (EDA)

Comprehensive EDA with automated file export to results directory.

In [4]:
# Code Chunk ID: CHUNK_2_1_EDA - COMPREHENSIVE EDA
# ============================================================================
# Run comprehensive EDA with single function call
# ============================================================================

from src.data.eda import run_comprehensive_eda

eda_results = run_comprehensive_eda(
    data=data,
    target_column=target_column,
    dataset_identifier=DATASET_IDENTIFIER,
    dataset_name=NOTEBOOK_CONFIG.get('dataset_name'),
    categorical_columns=categorical_columns,
    verbose=True
)

# Update categorical_columns from EDA results (in case auto-detected)
categorical_columns = eda_results['categorical_columns']

print(f"\nEDA Results Summary:")
print(f"  Files generated: {len(eda_results['files_generated'])}")
print(f"  Categorical columns: {categorical_columns}")
print(f"  Balance ratio: {eda_results.get('balance_ratio', 'N/A')}")

COMPREHENSIVE EXPLORATORY DATA ANALYSIS
Dataset: Pakistani Diabetes Dataset
Target column: outcome
Results path: results/pakistani-diabetes-dataset/2026-03-16/Section-2

[1/6] Dataset Overview...
   Dataset Name.................. Pakistani Diabetes Dataset
   Shape......................... 912 rows x 19 columns
   Memory Usage.................. 0.13 MB
   Total Missing Values.......... 0
   Missing Percentage............ 0.00%
   Duplicate Rows................ 2
   Numeric Columns............... 19
   Categorical Columns........... 0

[2/6] Column Analysis...
   Saved: column_analysis.csv (19 columns)

[3/6] Target Variable Analysis...
   Saved: target_analysis.csv
   Saved: target_balance_metrics.csv
   Balance Ratio: 0.877 (Balanced)

[4/6] Feature Distributions...
   Saved: 3 distribution file(s) (18 features)

[5/6] Correlation Analysis...
   Saved: correlation_heatmap.png
   Saved: correlation_matrix.csv
   Saved: target_correlations.csv (18 features)

[6/6] Configuration Validati

## 3 Demo All Models with Default Parameters

### 3.1 Batch Model Training

In [5]:
# Code Chunk ID: CHUNK_3_1_BATCH
# ============================================================================
# SECTION 3.1 - BATCH MODEL TRAINING
# Train all models configured in NOTEBOOK_CONFIG['models_to_run']
# ============================================================================

from src.models.batch_training import train_models_batch, extract_synthetic_data_to_globals

# Run batch training for all configured models (checkpoint resumes completed models)
training_results = train_models_batch(
    data=data,
    models_to_run=models_to_run,
    target_column=target_column,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    verbose=True,
    checkpoint=checkpoint
)

# Extract synthetic_data_* variables for Section 3.2 compatibility
created_vars = extract_synthetic_data_to_globals(training_results, globals())
print(f"\nCreated variables: {created_vars}")

# Also create model_* variables for backward compatibility
for model_name, result in training_results.items():
    if result['status'] == 'success' and result.get('model') is not None:
        globals()[f'{model_name}_model'] = result['model']


BATCH MODEL TRAINING
Models to train: 7
Dataset shape: (912, 19)
Target column: outcome
Samples to generate: 912
GPU available: NVIDIA A10G (22.1 GB)
Device assignments:
  - CTGAN: cuda
  - TVAE: cuda
  - CopulaGAN: cuda
  - CTABGAN: cuda
  - CTABGAN+: cuda
  - PATE-GAN: cuda
  - MEDGAN: cuda


[1/7] Training CTGAN...
--------------------------------------------------
  Device: cuda
  Training CTGAN...


Gen. (-1.38) | Discrim. (-0.14): 100%|██████████| 300/300 [00:10<00:00, 27.86it/s]


  Generating 912 synthetic samples...
  [OK] CTGAN completed in 18.19s
  Synthetic data shape: (912, 19)

[2/7] Training TVAE...
--------------------------------------------------
  Device: cuda
  Training TVAE...
  Generating 912 synthetic samples...
[OK] Target integrity functions loaded from src/data/target_integrity.py
  [OK] TVAE completed in 12.79s
  Synthetic data shape: (912, 19)

[3/7] Training CopulaGAN...
--------------------------------------------------
  Device: cuda
  Training CopulaGAN...
  Generating 912 synthetic samples...
  [OK] CopulaGAN completed in 11.87s
  Synthetic data shape: (912, 19)

[4/7] Training CTABGAN...
--------------------------------------------------
  Device: cuda
  Training CTABGAN...


100%|██████████| 300/300 [00:22<00:00, 13.36it/s]


Finished training in 24.94706106185913  seconds.
  Generating 912 synthetic samples...
  [OK] CTABGAN completed in 25.02s
  Synthetic data shape: (912, 19)

[5/7] Training CTABGAN+...
--------------------------------------------------
  Device: cuda
  Training CTABGAN+...


100%|██████████| 400/400 [00:54<00:00,  7.39it/s]


Finished training in 55.75640106201172  seconds.
  Generating 912 synthetic samples...
  [OK] CTABGAN+ completed in 55.85s
  Synthetic data shape: (912, 19)

[6/7] Training PATE-GAN...
--------------------------------------------------
  Device: cuda
  Training PATE-GAN...
  Generating 912 synthetic samples...
  [OK] PATE-GAN completed in 11.23s
  Synthetic data shape: (912, 19)

[7/7] Training MEDGAN...
--------------------------------------------------
  Device: cuda
  Training MEDGAN...
  Generating 912 synthetic samples...
  [OK] MEDGAN completed in 13.04s
  Synthetic data shape: (912, 19)

BATCH TRAINING SUMMARY
Total models: 7
Successful: 7
Failed: 0

Successful models:
  - CTGAN: 18.19s
  - TVAE: 12.79s
  - CopulaGAN: 11.87s
  - CTABGAN: 25.02s
  - CTABGAN+: 55.85s
  - PATE-GAN: 11.23s
  - MEDGAN: 13.04s

Created variables: ['synthetic_data_ctgan', 'synthetic_data_tvae', 'synthetic_data_copulagan', 'synthetic_data_ctabgan', 'synthetic_data_ctabganplus', 'synthetic_data_pategan',

### 3.2 Batch Evaluation

In [6]:
# Code Chunk ID: CHUNK_3_2_0_A
# ============================================================================
# SECTION 3.2 - BATCH EVALUATION FOR ALL TRAINED MODELS
# ============================================================================

print("SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION")
print("=" * 60)

if checkpoint.exists('section_3.2'):
    section3_results = checkpoint.load('section_3.2')['results']
    print("[RESUME] Loaded Section 3.2 evaluation from checkpoint")
else:
    section3_results = evaluate_all_available_models(
        section_number=3,
        scope=globals(),
        models_to_evaluate=None,
        real_data=None,
        target_col=None,
        protected_col=NOTEBOOK_CONFIG.get("protected_col")
    )
    checkpoint.save('section_3.2', {'results': section3_results})

if section3_results:
    print(f"\nSECTION 3 BATCH EVALUATION COMPLETED!")
    print(f"Evaluated {len(section3_results)} models successfully")
    
    # Show ranking by quality score
    best_models = []
    for model_name, results in section3_results.items():
        if 'error' not in results:
            quality_score = results.get('overall_quality_score', 0)
            best_models.append((model_name, quality_score))
    
    if best_models:
        best_models.sort(key=lambda x: x[1], reverse=True)
        print(f"\nRANKING BY QUALITY SCORE:")
        for i, (model, score) in enumerate(best_models, 1):
            print(f"   {i}. {model}: {score:.3f}")
else:
    print("\nNo models available for evaluation")

SECTION 3.2 - COMPREHENSIVE BATCH EVALUATION
[OK] Batch evaluation functions loaded from src/evaluation/batch.py
[SEARCH] UNIFIED BATCH EVALUATION - SECTION 3
[INFO] Dataset: pakistani-diabetes-dataset
[INFO] Target column: outcome
[INFO] Protected column: gender
[INFO] MIA evaluation: OFF
[INFO] Variable pattern: standard
[INFO] Found 7 trained models:
   [OK] CTGAN
   [OK] CTABGAN
   [OK] CTABGANPLUS
   [OK] CopulaGAN
   [OK] TVAE
   [OK] MEDGAN
   [OK] PATEGAN

==================== EVALUATING CTGAN ====================
[SEARCH] CTGAN - COMPREHENSIVE DATA QUALITY EVALUATION
[FOLDER] Output directory: results/pakistani-diabetes-dataset/2026-03-16/Section-3/CTGAN

[1] STATISTICAL SIMILARITY
------------------------------
   [CHART] Average Statistical Similarity: 0.801

[2] PCA COMPARISON ANALYSIS WITH OUTCOME COLOR-CODING
--------------------------------------------------
   [CHART] PCA comparison plot saved: pca_comparison_with_outcome.png
   [CHART] PCA Overall Similarity: 0.030
   

## 4 Staged Hyperparameter Optimization

This section uses a staged approach to hyperparameter optimization:

- **Smoke mode** (`tuning_mode="smoke"`): Runs 10 pilot trials per model, then displays
  time-budget recommendations (how many trials fit in 1 hour, how long 20 trials would take).
  Section 5 uses the pilot results directly.
- **Full mode** (`tuning_mode="full"`): Runs a pilot phase, displays time estimates, then
  presents three continuation strategies in cell 4.3.  The user reviews the estimates and
  **uncomments one option** before running the cell.

### 4.1 Configuration

Create the `StagedOptimizationManager`.  `TUNING_MODE` and `PILOT_TRIALS` are derived
from `NOTEBOOK_CONFIG` so the same code works for both smoke and full runs.

In [7]:
# Code Chunk ID: CHUNK_4_1_CONFIG
# ============================================================================
# SECTION 4.1 - STAGED OPTIMIZATION CONFIGURATION
# ============================================================================

from src.models.staged_optimization import (
    StagedOptimizationManager,
    StagedOptimizationConfig,
    flush_previous_runs
)

# Flush optimization studies if FRESH_START is set
if FRESH_START:
    flush_previous_runs(DATASET_IDENTIFIER)

# Derive tuning mode and pilot size from NOTEBOOK_CONFIG
TUNING_MODE = NOTEBOOK_CONFIG.get("tuning_mode", "smoke")
PILOT_TRIALS = 10 if TUNING_MODE == "smoke" else NOTEBOOK_CONFIG.get("n_trials_smoke", 10)

# Configure staged optimization
staged_config = StagedOptimizationConfig(
    pilot_trials=PILOT_TRIALS,
    verbose_level=1,           # 0=silent, 1=concise, 2=detailed
    persistence_dir=f"results/{DATASET_IDENTIFIER}/optimization_studies",
    run_mode=RUN_MODE,         # "debug" or "full"
    random_state=42,
    continue_on_error=True
)

# Create the manager
manager = StagedOptimizationManager(
    data=data,
    target_column=target_column,
    categorical_columns=categorical_columns,
    dataset_identifier=DATASET_IDENTIFIER,
    config=staged_config
)

print("Staged Optimization Manager created!")
print(f"  Tuning mode: {TUNING_MODE}")
print(f"  Pilot trials: {staged_config.pilot_trials}")
print(f"  Run mode: {staged_config.run_mode}")
print(f"  Persistence dir: {staged_config.persistence_dir}")
print(f"  FRESH_START: {FRESH_START}")

[FLUSH] No previous studies found in results/pakistani-diabetes-dataset/optimization_studies — starting clean
Staged Optimization Manager created!
  Tuning mode: full
  Pilot trials: 15
  Run mode: full
  Persistence dir: results/pakistani-diabetes-dataset/optimization_studies
  FRESH_START: True


### 4.2 Run Pilot Phase

Run a pilot phase to establish time estimates.

- **Smoke mode**: 10 trials + smoke recommendations table (trials in 1 hr, time for 20 trials).
- **Full mode**: Pilot trials + time estimates for planning continuation.

In [8]:
# Code Chunk ID: CHUNK_4_2_PILOT
# ============================================================================
# SECTION 4.2 - RUN PILOT PHASE
# ============================================================================

# Run pilot phase (uses PILOT_TRIALS from cell 4.1)
pilot_states = manager.run_pilot(
    models_to_run=models_to_run
)

# Display time estimates
print("\n" + "="*60)
print("PILOT PHASE COMPLETE - TIME ESTIMATES")
print("="*60)

time_estimates = manager.get_time_estimates()
if len(time_estimates) > 0:
    print(time_estimates.to_string(index=False))
else:
    print("No time estimates available")

# Show best scores so far
print("\nBest scores after pilot:")
for model_name, state in pilot_states.items():
    print(f"  {model_name}: {state.best_score:.4f} ({state.total_trials} trials)")

# Smoke mode: show budget recommendations
if TUNING_MODE == "smoke":
    print("\n" + "="*60)
    print("SMOKE MODE - TIME BUDGET RECOMMENDATIONS")
    print("="*60)
    smoke_recs = manager.get_smoke_recommendations()
    print(smoke_recs.to_string(index=False))
    print("\nTo run a full optimization, set tuning_mode='full' in NOTEBOOK_CONFIG")
    print("and use the recommended_pilot column to guide n_trials_smoke.")

[I 2026-03-16 20:48:35,401] A new study created in memory with name: ctgan_hpo_pakistani-diabetes-dataset



STAGED OPTIMIZATION - PILOT PHASE
Models to optimize: 7
Pilot trials per model: 15
Dataset identifier: pakistani-diabetes-dataset


[PILOT] Optimizing CTGAN...
--------------------------------------------------


Gen. (-0.83) | Discrim. (-0.49): 100%|██████████| 500/500 [00:31<00:00, 15.89it/s]


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5353
[OK] TRTS Evaluation: 2 scenarios, Average: 0.5603
[CHART] Combined Score: 0.5453 (Similarity: 0.5353, Accuracy: 0.5603)
[ctgan] Trial 1: Combined Score: 0.5453 (Similarity: 0.5353, Accuracy: 0.5603) Best Score so far: 0.5453


Gen. (-0.91) | Discrim. (0.08): 100%|██████████| 250/250 [00:16<00:00, 14.92it/s] 


...

Finished training in 82.225656747818  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6006
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9704
[CHART] Combined Score: 0.7485 (Similarity: 0.6006, Accuracy: 0.9704)
[ctabganplus] Trial 13: Combined Score: 0.7485 (Similarity: 0.6006, Accuracy: 0.9704) Best Score so far: 0.7720


100%|██████████| 1000/1000 [01:04<00:00, 15.57it/s]


Finished training in 65.81099390983582  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5726
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9709
[CHART] Combined Score: 0.7319 (Similarity: 0.5726, Accuracy: 0.9709)
[ctabganplus] Trial 14: Combined Score: 0.7319 (Similarity: 0.5726, Accuracy: 0.9709) Best Score so far: 0.7720


100%|██████████| 500/500 [03:37<00:00,  2.30it/s]


Finished training in 218.92921113967896  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5711
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9731
[CHART] Combined Score: 0.7319 (Similarity: 0.5711, Accuracy: 0.9731)
[ctabganplus] Trial 15: Combined Score: 0.7319 (Similarity: 0.5711, Accuracy: 0.9731) Best Score so far: 0.7720
  [OK] CTABGAN+: 15 trials in 1996.4s
  Best score: 0.7720

[PILOT] Optimizing PATE-GAN...
--------------------------------------------------
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.3406
[OK] TRTS Evaluation: 2 scenarios, Average: 0.2336
[CHART] Combined Score: 0.2978 (Similarity: 0.3406, Accuracy: 0.2336)
[pategan] Trial 1: Combined Score: 0.2978 (Similarity: 0.3406, Accuracy: 0.2336) Best Score so far: 0.2978
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity A

In [9]:
# ============================================================================
# DIMINISHING RETURNS ANALYSIS
# ============================================================================
print("DIMINISHING RETURNS ANALYSIS")
print("="*60)

convergence_report = manager.get_diminishing_returns_report()

if len(convergence_report) > 0:
    print(convergence_report.to_string(index=False))
    
    print("\nInterpretation:")
    print("-" * 40)
    for _, row in convergence_report.iterrows():
        print(f"  {row['model']}: {row['recommendation']}")
        if row['has_plateaued']:
            print(f"    -> Model has plateaued at score {row['best_score']:.4f}")
        else:
            print(f"    -> Improvement rate: {row['improvement_rate']:.4f}")
else:
    print("No convergence data available")

DIMINISHING RETURNS ANALYSIS
      model  trials  best_score  improvement_rate  total_improvement  has_plateaued                         recommendation
      ctgan      15    0.696737          0.025399           0.151440          False Consider stopping - minor improvements
       tvae      15    0.794947          0.000000           0.014303           True                 Stop - plateau reached
  copulagan      15    0.753792          0.000000           0.164907           True                 Stop - plateau reached
    ctabgan      15    0.755414          0.005323           0.025698           True                 Stop - plateau reached
ctabganplus      15    0.771954          0.000000           0.000000           True                 Stop - plateau reached
    pategan      15    0.354659          0.000000           0.056907           True                 Stop - plateau reached
     medgan      15    0.385176          0.000000           0.048344           True                 Stop - pla

### 4.3 Continuation (Full Mode Only)

**Full mode only** - Review the pilot results and time estimates above, then
uncomment **one** of the three options below and adjust the values before running.

- **Option (i)**: Common trial count for all models
- **Option (ii)**: Per-model trial counts
- **Option (iii)**: Time-based budget (minutes per model)

Models not in `models_to_run` are silently ignored, so listing all 8 is safe.

In [10]:
# Code Chunk ID: CHUNK_4_3_CONTINUE
# ============================================================================
# SECTION 4.3 - CONTINUATION (Full mode only - choose ONE option)
# ============================================================================

if TUNING_MODE == "smoke":
    print("[SMOKE MODE] Skipping continuation - using pilot results for Section 5.")

else:
    # Choose ONE option below, then run this cell.

    # OPTION (i): Common trials for all models
    # continuation_states = manager.continue_optimization(additional_trials=30)

    # OPTION (ii): Per-model trials - adjust counts per model
    continuation_states = manager.continue_optimization(
        trials_per_model={
            "ctgan": 30,
            # "tvae": 30,
            # "copulagan": 30,
            # "ctabgan": 30,
            "ctabganplus": 15,
            # "ganeraid": 30,
            # "pategan": 30,
            "medgan": 50,
        }
    )

    # OPTION (iii): Time-based budget - minutes per model
    # continuation_states = manager.continue_optimization(
    #     time_budget_minutes={
    #         "ctgan": 60,
    #         "tvae": 60,
    #         "copulagan": 60,
    #         "ctabgan": 60,
    #         "ctabganplus": 60,
    #         "ganeraid": 60,
    #         "pategan": 60,
    #         "medgan": 60,
    #     }
    # )

    print("[FULL MODE] Continuation completed.")


STAGED OPTIMIZATION - CONTINUATION PHASE
  ctgan: 30 additional trials
  ctabganplus: 15 additional trials
  medgan: 50 additional trials


[CONTINUE] Optimizing CTGAN...
--------------------------------------------------
  Resuming from 15 existing trials


Gen. (-0.50) | Discrim. (-0.26): 100%|██████████| 650/650 [00:31<00:00, 20.50it/s]


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5402
[OK] TRTS Evaluation: 2 scenarios, Average: 0.6387
[CHART] Combined Score: 0.5796 (Similarity: 0.5402, Accuracy: 0.6387)
[ctgan] Trial 16: Combined Score: 0.5796 (Similarity: 0.5402, Accuracy: 0.6387) Best Score so far: 0.6967


Gen. (-0.70) | Discrim. (-0.11): 100%|██████████| 850/850 [00:41<00:00, 20.32it/s]


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5503
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8755
[CHART] Combined Score: 0.6804 (Similarity: 0.5503, Accuracy: 0.8755)
[ctgan] Trial 17: Combined Score: 0.6804 (Similarity: 0.5503, Accuracy: 0.8755) Best Score so far: 0.6967


...

Finished training in 464.1312379837036  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6017
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9836
[CHART] Combined Score: 0.7544 (Similarity: 0.6017, Accuracy: 0.9836)
[ctabganplus] Trial 28: Combined Score: 0.7544 (Similarity: 0.6017, Accuracy: 0.9836) Best Score so far: 0.7746


100%|██████████| 950/950 [09:06<00:00,  1.74it/s]


Finished training in 548.0998973846436  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6199
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9841
[CHART] Combined Score: 0.7656 (Similarity: 0.6199, Accuracy: 0.9841)
[ctabganplus] Trial 29: Combined Score: 0.7656 (Similarity: 0.6199, Accuracy: 0.9841) Best Score so far: 0.7746


100%|██████████| 850/850 [08:15<00:00,  1.72it/s]


Finished training in 497.1794126033783  seconds.
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5900
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9830
[CHART] Combined Score: 0.7472 (Similarity: 0.5900, Accuracy: 0.9830)
[ctabganplus] Trial 30: Combined Score: 0.7472 (Similarity: 0.5900, Accuracy: 0.9830) Best Score so far: 0.7746
  [OK] CTABGAN+: 15 trials in 5806.9s
  Best score: 0.7746

[CONTINUE] Optimizing MEDGAN...
--------------------------------------------------
  Resuming from 15 existing trials
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.3864
[PRUNED] Trial pruned after similarity calculation (score: 0.3864)
[medgan] Trial 16: Combined Score: 0.3864 (Similarity: 0.0000, Accuracy: 0.0000) Best Score so far: 0.3852
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 vali

In [11]:
# ============================================================================
# DIMINISHING RETURNS ANALYSIS (post-continuation)
# ============================================================================

if TUNING_MODE == "full":
    print("DIMINISHING RETURNS ANALYSIS")
    print("="*60)

    convergence_report = manager.get_diminishing_returns_report()

    if len(convergence_report) > 0:
        print(convergence_report.to_string(index=False))

        print("\nInterpretation:")
        print("-" * 40)
        for _, row in convergence_report.iterrows():
            print(f"  {row['model']}: {row['recommendation']}")
            if row['has_plateaued']:
                print(f"    -> Model has plateaued at score {row['best_score']:.4f}")
            else:
                print(f"    -> Improvement rate: {row['improvement_rate']:.4f}")
    else:
        print("No convergence data available")
else:
    print("[SMOKE MODE] Skipping post-continuation analysis.")

DIMINISHING RETURNS ANALYSIS
      model  trials  best_score  improvement_rate  total_improvement  has_plateaued         recommendation
      ctgan      45    0.703924          0.000000           0.158627           True Stop - plateau reached
       tvae      15    0.794947          0.000000           0.014303           True Stop - plateau reached
  copulagan      15    0.753792          0.000000           0.164907           True Stop - plateau reached
    ctabgan      15    0.755414          0.005323           0.025698           True Stop - plateau reached
ctabganplus      30    0.774570          0.000000           0.002616           True Stop - plateau reached
    pategan      15    0.354659          0.000000           0.056907           True Stop - plateau reached
     medgan      65    0.441779          0.000000           0.104947           True Stop - plateau reached

Interpretation:
----------------------------------------
  ctgan: Stop - plateau reached
    -> Model has plateaue

### 4.5 Additional Batches (Full Mode, Optional)

If the diminishing returns analysis suggests continuing, uncomment one option below.
You can re-run this cell multiple times.

In [ ]:
# Code Chunk ID: CHUNK_4_5_ADDITIONAL

# ============================================================================
# SECTION 4.5 - ADDITIONAL BATCHES (Full mode only - optional, can repeat)
# ============================================================================

if TUNING_MODE == "smoke":
    print("[SMOKE MODE] Skipping additional batches.")
else:
    # ---- Uncomment ONE option below, adjust values, then run this cell ----

    # OPTION (i): Common trials for all models
    # additional_states = manager.continue_optimization(additional_trials=20)

    # OPTION (ii): Per-model trials - adjust counts per model
    # additional_states = manager.continue_optimization(
    #     trials_per_model={
    #         'ctgan': 20,
    #         'tvae': 20,
    #         'copulagan': 20,
    #         'ctabgan': 20,
    #         'ctabganplus': 20,
    #         'ganeraid': 20,
    #         'pategan': 20,
    #         'medgan': 20,
    #     }
    # )

    # OPTION (iii): Time-based budget - minutes per model
    additional_states = manager.continue_optimization(
        time_budget_minutes={
            'ctgan': 30,
            'tvae': 30,
            # 'copulagan': 30,
            # 'ctabgan': 30,
            'ctabganplus': 30,
            # 'ganeraid': 30,
            # 'pategan': 30,
            # 'medgan': 30,
        }
    )

    # print("\nUpdated convergence report:")
    # print(manager.get_diminishing_returns_report().to_string(index=False))

    print("Cell skipped - uncomment an option above to run additional batches")



STAGED OPTIMIZATION - CONTINUATION PHASE
  ctgan: 47 additional trials
  tvae: 87 additional trials
  ctabganplus: 6 additional trials


[CONTINUE] Optimizing CTGAN...
--------------------------------------------------
  Resuming from 45 existing trials


Gen. (-0.84) | Discrim. (-0.36): 100%|██████████| 800/800 [01:58<00:00,  6.76it/s]


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5241
[OK] TRTS Evaluation: 2 scenarios, Average: 0.7758
[CHART] Combined Score: 0.6248 (Similarity: 0.5241, Accuracy: 0.7758)
[ctgan] Trial 46: Combined Score: 0.6248 (Similarity: 0.5241, Accuracy: 0.7758) Best Score so far: 0.7039


Gen. (-1.21) | Discrim. (0.37): 100%|██████████| 1000/1000 [02:41<00:00,  6.21it/s]


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5273
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8991
[CHART] Combined Score: 0.6761 (Similarity: 0.5273, Accuracy: 0.8991)
[ctgan] Trial 47: Combined Score: 0.6761 (Similarity: 0.5273, Accuracy: 0.8991) Best Score so far: 0.7039


...

[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5296
[OK] TRTS Evaluation: 2 scenarios, Average: 0.9052
[CHART] Combined Score: 0.6798 (Similarity: 0.5296, Accuracy: 0.9052)
[ctgan] Trial 90: Combined Score: 0.6798 (Similarity: 0.5296, Accuracy: 0.9052) Best Score so far: 0.7039


Gen. (-0.91) | Discrim. (0.30): 100%|██████████| 100/100 [00:12<00:00,  8.12it/s]


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5567
[OK] TRTS Evaluation: 2 scenarios, Average: 0.6102
[CHART] Combined Score: 0.5781 (Similarity: 0.5567, Accuracy: 0.6102)
[ctgan] Trial 91: Combined Score: 0.5781 (Similarity: 0.5567, Accuracy: 0.6102) Best Score so far: 0.7039


Gen. (-1.49) | Discrim. (0.13): 100%|██████████| 1000/1000 [02:01<00:00,  8.26it/s]


[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.5377
[OK] TRTS Evaluation: 2 scenarios, Average: 0.8898
[CHART] Combined Score: 0.6785 (Similarity: 0.5377, Accuracy: 0.8898)
[ctgan] Trial 92: Combined Score: 0.6785 (Similarity: 0.5377, Accuracy: 0.8898) Best Score so far: 0.7039
  [OK] CTGAN: 47 trials in 7048.4s
  Best score: 0.7039

[CONTINUE] Optimizing TVAE...
--------------------------------------------------
  Resuming from 15 existing trials
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6455
[PRUNED] Trial pruned after similarity calculation (score: 0.6455)
[tvae] Trial 16: Combined Score: 0.6455 (Similarity: 0.0000, Accuracy: 0.0000) Best Score so far: 0.7949
[TARGET] Enhanced objective function using target column: 'outcome'
[OK] Similarity Analysis: 19/19 valid metrics, Average: 0.6318
[PRUNED] Trial pruned after similar

### 4.6 Save Best Parameters

In [17]:
# Code Chunk ID: CHUNK_4_6_SAVE
# ============================================================================
# SECTION 4.6 - SAVE BEST PARAMETERS TO CSV
# ============================================================================

from src.utils.parameters import save_best_parameters_to_csv

# Extract studies from manager to globals for saving
for model_name, state in manager.model_states.items():
    if state.study is not None:
        globals()[f"{model_name}_study"] = state.study

# Save all best parameters to CSV for Section 5
save_result = save_best_parameters_to_csv(
    scope=globals(),
    section_number=4,
    dataset_identifier=DATASET_IDENTIFIER
)

print(f"\nParameters saved: {save_result['success']}")
print(f"Files: {save_result.get('files_saved', [])}")

[SAVE] SAVING BEST PARAMETERS FROM SECTION 4
[FOLDER] Target directory: results/pakistani-diabetes-dataset/2026-03-16/Section-4

[CHART] Processing CTGAN parameters...
[OK] Found CTGAN: 10 parameters, score: 0.7039

[CHART] Processing CTAB-GAN parameters...
[OK] Found CTAB-GAN: 2 parameters, score: 0.7554

[CHART] Processing CTAB-GAN+ parameters...
[OK] Found CTAB-GAN+: 2 parameters, score: 0.7746

[CHART] Processing GANerAid parameters...
[WARNING]  GANerAid: Study variable 'ganeraid_study' not found

[CHART] Processing CopulaGAN parameters...
[OK] Found CopulaGAN: 6 parameters, score: 0.7538

[CHART] Processing TVAE parameters...
[OK] Found TVAE: 8 parameters, score: 0.8044

[CHART] Processing PATE-GAN parameters...
[OK] Found PATE-GAN: 10 parameters, score: 0.3547

[CHART] Processing MEDGAN parameters...
[OK] Found MEDGAN: 11 parameters, score: 0.4418

[OK] Parameters saved: results/pakistani-diabetes-dataset/2026-03-16/Section-4/best_parameters.csv
   - Total parameter entries: 61


## 5 Final Model Comparison with Best Parameters

### 5.1 Train All Models with Best Parameters from Section 4

In [18]:
# Code Chunk ID: CHUNK_5_1_BATCH
# ============================================================================
# SECTION 5.1 - BATCH TRAINING WITH BEST PARAMETERS
# ============================================================================

from src.models.batch_training import train_models_batch_with_best_params

# Train all models with best parameters from Section 4 (checkpoint resumes completed models)
section5_results = train_models_batch_with_best_params(
    data=data,
    target_column=target_column,
    models_to_run=models_to_run,
    categorical_columns=categorical_columns,
    n_samples=len(data),
    random_state=42,
    section_number=4,
    dataset_identifier=DATASET_IDENTIFIER,
    scope=globals(),
    verbose=True,
    checkpoint=checkpoint
)

# Show summary of created variables
final_vars = [k for k in globals().keys() if k.endswith('_final')]
print(f"\nFinal synthetic data variables: {sorted(final_vars)}")


SECTION 5.1: BATCH TRAINING WITH BEST PARAMETERS
Models to train: 7
Dataset shape: (912, 19)
Target column: outcome
Samples to generate: 912
Loading parameters from: Section 4
GPU available: NVIDIA A10G (22.1 GB)
Device assignments:
  - CTGAN: cuda
  - TVAE: cuda
  - CopulaGAN: cuda
  - CTABGAN: cuda
  - CTABGAN+: cuda
  - PATE-GAN: cuda
  - MEDGAN: cuda

[1/3] Loading best parameters from Section 4...
[LOAD] LOADING BEST PARAMETERS FROM SECTION 4
[FOLDER] Looking for: results/pakistani-diabetes-dataset/2026-03-16/Section-4/best_parameters.csv
[OK] Found parameter CSV file
[OK] Loaded CTGAN: 10 parameters
[OK] Loaded CTAB-GAN: 2 parameters
[OK] Loaded CTAB-GAN+: 2 parameters
[OK] Loaded CopulaGAN: 6 parameters
[OK] Loaded TVAE: 8 parameters
[OK] Loaded PATE-GAN: 10 parameters
[OK] Loaded MEDGAN: 11 parameters

[LOAD] Parameter loading completed!
[SEARCH] Source: CSV file
[CHART] Models loaded: 7
   - ctgan: 10 parameters
   - ctabgan: 2 parameters
   - ctabganplus: 2 parameters
   - c

### 5.2 Batch Evaluation of Optimized Models

In [19]:
# Code Chunk ID: CHUNK_5_2_0_A
# ============================================================================
# SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
# ============================================================================

print("SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION")
print("=" * 80)

from setup import evaluate_section5_optimized_models

if checkpoint.exists('section_5.2'):
    section5_batch_results = checkpoint.load('section_5.2')['results']
    print("[RESUME] Loaded Section 5.2 evaluation from checkpoint")
    print(f"Models processed: {section5_batch_results['models_processed']}")
    print(f"Results exported to: {section5_batch_results['results_dir']}")
else:
    try:
        section5_batch_results = evaluate_section5_optimized_models(
            section_number=5,
            scope=globals(),
            target_column=TARGET_COLUMN,
            protected_col=NOTEBOOK_CONFIG.get("protected_col"),
            compute_mia=True
        )
        checkpoint.save('section_5.2', {'results': section5_batch_results})
        
        print("\n" + "="*80)
        print("SECTION 5.2 OPTIMIZED MODELS BATCH EVALUATION COMPLETED!")
        print("="*80)
        print(f"Models processed: {section5_batch_results['models_processed']}")
        print(f"Results exported to: {section5_batch_results['results_dir']}")
        
    except Exception as e:
        print(f"Section 5.2 batch evaluation failed: {e}")
        import traceback
        traceback.print_exc()

SECTION 5.2 - OPTIMIZED MODELS BATCH EVALUATION
[RESUME] Loaded Section 5.2 evaluation from checkpoint
Models processed: 7
Results exported to: results/pakistani-diabetes-dataset/2026-03-16/Section-5


### 5.3 Final Summary

In [21]:
# Code Chunk ID: CHUNK_5_3_SUMMARY
# ============================================================================
# SECTION 5.3 - FINAL SUMMARY
# ============================================================================

print("=" * 80)
print("SYNTHETIC DATA GENERATION - FINAL SUMMARY")
print("=" * 80)
print(f"\nDataset: {DATASET_IDENTIFIER}")
print(f"Session: {SESSION_TIMESTAMP}")
print(f"\nResults directories:")
print(f"  Section 2 (EDA): {get_results_path(DATASET_IDENTIFIER, 2)}")
print(f"  Section 3 (Demo): {get_results_path(DATASET_IDENTIFIER, 3)}")
print(f"  Section 4 (HPO): {get_results_path(DATASET_IDENTIFIER, 4)}")
print(f"  Section 5 (Final): {get_results_path(DATASET_IDENTIFIER, 5)}")

# Show staged optimization summary
if 'manager' in dir() and manager is not None:
    print(f"\nStaged Optimization Summary:")
    print("-" * 60)
    summary = manager.get_best_params_summary()
    if len(summary) > 0:
        print(summary[['model', 'best_score', 'total_trials', 'status']].to_string(index=False))

# Show final model performance
if 'section5_results' in dir() and section5_results:
    print(f"\nFinal Model Performance (with optimized parameters):")
    print("-" * 60)
    
    scores = []
    for model_name, result in section5_results.items():
        if result['status'] == 'success':
            score = result.get('objective_score', 0)
            time_taken = result.get('training_time', 0)
            scores.append((model_name, score, time_taken))
    
    # Sort by score descending
    scores.sort(key=lambda x: x[1], reverse=True)
    
    for i, (model, score, time_taken) in enumerate(scores, 1):
        print(f"  {i}. {model.upper()}: score={score:.4f}, time={time_taken:.1f}s")
    
    if scores:
        best_model = scores[0][0]
        print(f"\nBest performing model: {best_model.upper()}")
        print(f"Best synthetic data variable: synthetic_{best_model}_final")

print("\n" + "=" * 80)
print("NOTEBOOK COMPLETE")
print("=" * 80)

SYNTHETIC DATA GENERATION - FINAL SUMMARY

Dataset: pakistani-diabetes-dataset
Session: 2026-03-16

Results directories:
  Section 2 (EDA): results/pakistani-diabetes-dataset/2026-03-16/Section-2
  Section 3 (Demo): results/pakistani-diabetes-dataset/2026-03-16/Section-3
  Section 4 (HPO): results/pakistani-diabetes-dataset/2026-03-16/Section-4
  Section 5 (Final): results/pakistani-diabetes-dataset/2026-03-16/Section-5

Staged Optimization Summary:
------------------------------------------------------------
      model  best_score  total_trials    status
      ctgan    0.703924            92 completed
       tvae    0.804446           102 completed
  copulagan    0.753792            15 completed
    ctabgan    0.755414            15 completed
ctabganplus    0.774570            36 completed
    pategan    0.354659            15 completed
     medgan    0.441779            65 completed

Final Model Performance (with optimized parameters):
-----------------------------------------------